In [8]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc, confusion_matrix
import pickle
import os

os.makedirs("../results/figures", exist_ok=True)
pd.set_option("display.float_format", "{:.3f}".format)

print("Librairies chargées ✓")

Librairies chargées ✓


In [9]:
# Chargement du dataset
dataset = pd.read_csv("../data/dataset.csv", parse_dates=["date"])

# Séparation train/test (même logique que ml_model.py)
FEATURES = [
    "trailingPE", "priceToBook", "debtToEquity",
    "returnOnEquity", "returnOnAssets", "profitMargins",
    "currentRatio", "floatShares", "momentum_1m",
    "momentum_3m", "momentum_6m", "volatility", "rel_strength",
]

train = dataset[dataset["date"] < "2022-01-01"]
test  = dataset[
    (dataset["date"] >= "2022-01-01") &
    (dataset["date"] <  "2023-01-01")
]

X_train = train[FEATURES]
y_train = train["label"]
X_test  = test[FEATURES]
y_test  = test["label"]

# Chargement du modèle sauvegardé
with open("../results/gradient_boosting.pkl", "rb") as f:
    gb_model = pickle.load(f)

print(f"Données d'entraînement : {X_train.shape}")
print(f"Données de test        : {X_test.shape}")
print(f"Modèle chargé          : Gradient Boosting ✓")

Données d'entraînement : (13577, 13)
Données de test        : (5500, 13)
Modèle chargé          : Gradient Boosting ✓


## Évaluation du modèle ML

On évalue ici le **Gradient Boosting** — notre meilleur modèle.On va visualiser trois choses :

1. **La courbe ROC** — mesure la capacité du modèle à distinguer les actions qui surperforment de celles qui sous-performent
2. **La matrice de confusion** — montre concrètement combien de bonnes et mauvaises prédictions le modèle fait
3. **L'importance des features** — quelles variables influencent le plus les décisions du modèle

Un modèle parfait aurait un AUC de 1.0 — un modèle aléatoire aurait 0.5. En finance, dépasser 0.55 est déjà considéré comme un signal exploitable.

In [10]:
# Probabilités prédites par le modèle
y_prob = gb_model.predict_proba(X_test)[:, 1]

# Calcul de la courbe ROC
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

# Graphique
fig = go.Figure()

# Courbe du modèle
fig.add_trace(go.Scatter(
    x=fpr, y=tpr,
    mode="lines",
    name=f"Gradient Boosting (AUC = {roc_auc:.3f})",
    line=dict(color="#1D9E75", width=2.5)
))

# Ligne du hasard (modèle aléatoire)
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode="lines",
    name="Modèle aléatoire (AUC = 0.500)",
    line=dict(color="#D85A30", width=1.5, dash="dash")
))

fig.update_layout(
    title="Courbe ROC — Gradient Boosting",
    xaxis_title="Taux de faux positifs",
    yaxis_title="Taux de vrais positifs",
    template="plotly_white",
    width=650, height=500,
    legend=dict(x=0.6, y=0.1)
)

fig.show()

fig.write_html("../results/figures/03_courbe_roc.html")
fig.write_image("../results/figures/03_courbe_roc.png")

print(f"AUC Score : {roc_auc:.3f}")

Resorting to unclean kill browser.


AUC Score : 0.462


In [11]:
# Prédictions avec seuil de 0.60
y_pred = (y_prob >= 0.60).astype(int)

# Calcul de la matrice
cm = confusion_matrix(y_test, y_pred)

# Graphique
fig = px.imshow(
    cm,
    text_auto=True,
    labels=dict(x="Prédit", y="Réel", color="Nombre"),
    x=["Sous-performe", "Surperforme"],
    y=["Sous-performe", "Surperforme"],
    color_continuous_scale=["white", "#1D9E75"],
    title="Matrice de confusion — Gradient Boosting",
    template="plotly_white",
    width=500, height=450
)

fig.update_layout(coloraxis_showscale=False)
fig.show()

fig.write_html("../results/figures/03_matrice_confusion.html")
fig.write_image("../results/figures/03_matrice_confusion.png")

Resorting to unclean kill browser.


In [12]:
# Récupération des importances depuis le modèle
importance = pd.Series(
    gb_model.feature_importances_,
    index=FEATURES
).sort_values(ascending=True)

fig = px.bar(
    x=importance.values,
    y=importance.index,
    orientation="h",
    title="Importance des features — Gradient Boosting",
    labels={"x": "Importance", "y": "Feature"},
    color=importance.values,
    color_continuous_scale=["#F5C4B3", "#1D9E75"],
    template="plotly_white",
    width=700, height=500
)

fig.update_layout(coloraxis_showscale=False)
fig.show()

fig.write_html("../results/figures/03_feature_importance.html")
fig.write_image("../results/figures/03_feature_importance.png")

Resorting to unclean kill browser.


In [13]:
# On réentraîne les trois modèles pour comparer
# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100, max_depth=5,
    random_state=42, class_weight="balanced"
)
rf_model.fit(X_train, y_train)

# Régression Logistique
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

lr_model = LogisticRegression(
    random_state=42, class_weight="balanced", max_iter=1000
)
lr_model.fit(X_train_scaled, y_train)

# Probabilités des trois modèles
prob_rf = rf_model.predict_proba(X_test)[:, 1]
prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]
prob_gb = gb_model.predict_proba(X_test)[:, 1]

# Courbes ROC des trois modèles
fig = go.Figure()

models = [
    ("Random Forest",         prob_rf, "#534AB7"),
    ("Régression Logistique", prob_lr, "#D85A30"),
    ("Gradient Boosting",     prob_gb, "#1D9E75"),
]

for name, prob, color in models:
    fpr, tpr, _ = roc_curve(y_test, prob)
    roc_auc     = auc(fpr, tpr)
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr,
        mode="lines",
        name=f"{name} (AUC = {roc_auc:.3f})",
        line=dict(color=color, width=2.5)
    ))

# Ligne du hasard
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode="lines",
    name="Modèle aléatoire",
    line=dict(color="gray", width=1.5, dash="dash")
))

fig.update_layout(
    title="Comparaison des modèles — Courbes ROC",
    xaxis_title="Taux de faux positifs",
    yaxis_title="Taux de vrais positifs",
    template="plotly_white",
    width=700, height=500,
    legend=dict(x=0.5, y=0.1)
)

fig.show()

fig.write_html("../results/figures/03_comparaison_modeles.html")
fig.write_image("../results/figures/03_comparaison_modeles.png")

c:\Users\HP\Desktop\Data Portfolio\Africa_ml_stock\venv\Lib\site-packages\choreographer\utils\_tmpfile.py:137: TmpDirWarning: The temporary directory could not be deleted, execution will continue. errors: [(WindowsPath('C:/Users/HP/AppData/Local/Temp/tmp_voxoo3n/CrashpadMetrics-active.pma'), PermissionError(13, 'Accès refusé')), (WindowsPath('C:/Users/HP/AppData/Local/Temp/tmp_voxoo3n'), OSError(41, 'Le répertoire n’est pas vide'))]
  warnings.warn(  # noqa: B028
Temporary dictory couldn't be removed manually.


In [14]:
# On visualise comment le modèle distribue ses probabilités
# entre les actions qui surperforment et celles qui sous-performent

prob_df = pd.DataFrame({
    "probabilité" : prob_gb,
    "réel"        : y_test.map({0: "Sous-performe", 1: "Surperforme"})
})

fig = px.histogram(
    prob_df,
    x="probabilité",
    color="réel",
    barmode="overlay",
    nbins=50,
    title="Distribution des probabilités prédites — Gradient Boosting",
    labels={
        "probabilité" : "Probabilité de surperformance",
        "réel"        : "Valeur réelle"
    },
    color_discrete_map={
        "Surperforme"  : "#1D9E75",
        "Sous-performe": "#D85A30"
    },
    opacity=0.7,
    template="plotly_white",
    width=750, height=450
)

fig.add_vline(
    x=0.55, line_dash="dash",
    line_color="gray",
    annotation_text="Seuil = 0.55"
)

fig.update_layout(legend_title="Valeur réelle")
fig.show()

fig.write_html("../results/figures/03_distribution_probabilites.html")
fig.write_image("../results/figures/03_distribution_probabilites.png")

## Limites identifiées et pistes d'amélioration

Les scores AUC obtenus (0.46 - 0.47) révèlent des limites claires qu'il est important de documenter honnêtement.

### Pourquoi le modèle peine à prédire ?

**1. Fondamentaux statiques** — on dispose d'un seul snapshot des ratios financiers pour toute la période 2019-2024. En réalité ces ratios changent chaque trimestre. Un modèle avec des données trimestrielles historiques serait bien plus performant.

**2. Univers restreint** — 22 actions c'est peu pour un modèle ML. Plus on a d'exemples diversifiés, mieux le modèle généralise. Un univers de 100+ actions donnerait de meilleurs résultats.

**3. Signaux faibles** — comme on l'a vu dans le notebook 02, aucune feature n'est fortement corrélée avec le label. Les marchés sont difficiles à prédire — c'est une réalité bien documentée en finance quantitative.

### Comment améliorer ?

- Récupérer des fondamentaux **trimestriels historiques** 
  via une API payante (Bloomberg, Refinitiv)
- Ajouter des **données macroéconomiques** : taux d'intérêt 
  sud-africains, prix des matières premières, taux de change ZAR
- Tester des modèles plus avancés : **LSTM** pour capturer 
  les séquences temporelles
- Élargir l'univers à **toutes les actions JSE** (~350 titres)